<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%85%83%E6%A8%A1%E5%9E%8B%E5%BB%BA%E7%AB%8B/composite_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from xgboost import XGBClassifier
xgb_url_model = XGBClassifier()
xgb_url_model.load_model('/content/url_xgb_v1.json')

In [3]:
!pip install joblib

In [4]:
import joblib
from sklearn.preprocessing import StandardScaler
scaler = joblib.load('/content/html_scaler.pkl')

In [5]:
%%capture
!pip install tensorflow

In [6]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
mlp_html_model = load_model('/content/html_mlp_v1.keras')

In [8]:
html_sca_col = ['links_in_tags', 'nb_hyperlinks']

In [7]:
from xgboost import XGBClassifier
xgb_ai_model = XGBClassifier()
xgb_ai_model.load_model('/content/ai_xgb_v1.json')

In [9]:
import pandas as pd

In [12]:
df_url = pd.read_csv('/content/phishing_dataset_expansion_forEmbeddingModule_url.csv')
df_html = pd.read_csv('/content/phishing_dataset_expansion_forEmbeddingModule_html.csv')
df_ai = pd.read_csv('/content/phishing_dataset_expansion_forEmbeddingModule_Gemini_text_success.csv')# .drop('visible_text', axis=1, inplace=True)

/tmp/ipykernel_2385/2391875751.py:3: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ai = pd.read_csv('/content/phishing_dataset_expansion_forEmbeddingModule_Gemini_text_success.csv')# .drop('visible_text', axis=1, inplace=True)


In [13]:
df_url.columns

Index(['url', 'target', 'length_url', 'length_hostname', 'ip', 'nb_dots',
       'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq',
       'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star',
       'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space',
       'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token',
       'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port',
       'tld_in_path', 'tld_in_subdomain', 'nb_subdomains',
       'abnormal_subdomain', 'prefix_suffix', 'path_extension',
       'length_words_raw', 'char_repeat', 'shortest_word_host',
       'shortest_word_path', 'longest_words_raw', 'longest_word_host',
       'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path'],
      dtype='object')

In [14]:
df_html.columns

Index(['url', 'target', 'phish_hints', 'domain_in_brand', 'nb_hyperlinks',
       'ratio_intHyperlinks', 'ratio_extHyperlinks', 'ratio_extRedirection',
       'ratio_extErrors', 'external_favicon', 'links_in_tags',
       'ratio_extMedia', 'safe_anchor', 'empty_title', 'domain_in_title',
       'domain_with_copyright', 'has_meta_refresh', 'has_js_redirect',
       'feature_extracted'],
      dtype='object')

In [15]:
df_ai.columns

Index(['url', 'target', 'visible_text', 'ai_status', 'fetch_status',
       'creates_urgency', 'uses_threats', 'requests_sensitive_info',
       'offers_unrealistic_rewards', 'has_spelling_grammar_errors',
       'impersonated_brand', 'has_valid_copyright_year',
       'is_content_login_focused', 'has_rich_navigation',
       'has_physical_address', 'has_phone_number', 'content_consistency_score',
       'language_professionalism_score', 'overall_phishing_likelihood_score'],
      dtype='object')

In [17]:
df_ai['text_length'] = df_ai['visible_text'].apply(lambda x: len(str(x)))

In [18]:
df_ai.drop('visible_text', axis=1, inplace=True)

In [19]:
df_c = pd.merge(df_url, df_html, on='url', how='inner')
df_con = pd.merge(df_c, df_ai, on='url', how='inner')

In [20]:
df_con = df_con[df_con['ai_status'] == 'AI_SUCCESS'] # df_con['gemini_status'] == 'OK' or

In [21]:
df_con['ai_status'].unique()

array(['AI_SUCCESS'], dtype=object)

In [22]:
df_con['creates_urgency'].unique()

array([False, True], dtype=object)

In [23]:
df_con['feature_extracted'].unique()

array([ 1.,  0., nan])

In [24]:
df_con = df_con[df_con['feature_extracted'] == 1]

In [25]:
df_con

,url,target_x,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,...,impersonated_brand,has_valid_copyright_year,is_content_login_focused,has_rich_navigation,has_physical_address,has_phone_number,content_consistency_score,language_professionalism_score,overall_phishing_likelihood_score,text_length
0,https://www.southbankmosaics.com,1,32,24,0,2,0,0,0,0,...,NaN,False,False,False,False,False,3.0,4.0,3.0,8934
1,https://www.uni-mainz.de,1,24,16,0,2,1,0,0,0,...,Johannes Gutenberg-Universität Mainz,False,False,True,True,False,8.0,10.0,2.0,3673
2,https://www.voicefmradio.co.uk,1,30,22,0,3,0,0,0,0,...,NaN,True,False,True,True,False,8.0,10.0,2.0,3095
5,https://www.globalreporting.org,1,31,23,0,2,0,0,0,0,...,NaN,True,False,True,False,False,9.0,10.0,2.0,3681
6,https://www.saffronart.com,1,26,18,0,2,0,0,0,0,...,Saffronart,True,False,True,False,False,10.0,10.0,2.0,1515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62709,https://www.perlen-reihe.at,1,27,19,0,2,1,0,0,0,...,Perlen-Reihe,False,False,True,True,True,8.0,10.0,2.0,553
63871,http://facebook.idiyanale.ph/,0,29,21,0,2,0,0,0,0,...,NaN,False,True,True,False,False,6.0,8.0,6.0,2613
73630,http://dhl-event.app/,0,21,13,0,1,1,0,0,0,...,NaN,False,False,False,False,False,3.0,8.0,5.0,297
83851,https://swisspassidsbb.cluster1.easy-hebergeme...,0,57,44,0,3,1,0,0,0,...,NaN,False,False,False,False,False,6.0,8.0,2.0,186


In [26]:
sca_col = ['links_in_tags', 'nb_hyperlinks']
df_con[sca_col] = scaler.fit_transform(df_con[sca_col])

In [27]:
df_con.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32704 entries, 0 to 84119
Data columns (total 83 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   url                                32704 non-null  object 
 1   target_x                           32704 non-null  int64  
 2   length_url                         32704 non-null  int64  
 3   length_hostname                    32704 non-null  int64  
 4   ip                                 32704 non-null  int64  
 5   nb_dots                            32704 non-null  int64  
 6   nb_hyphens                         32704 non-null  int64  
 7   nb_at                              32704 non-null  int64  
 8   nb_qm                              32704 non-null  int64  
 9   nb_and                             32704 non-null  int64  
 10  nb_or                              32704 non-null  int64  
 11  nb_eq                              32704 non-null  int64  


In [ ]:
df_con['ratio_intHyperlinks'] = df_con['ratio_intHyperlink']

KeyError: 'ratio_intHyperlink'

In [28]:

from sklearn.model_selection import train_test_split
X_url_test = df_con[['length_url', 'length_hostname', 'ip', 'nb_dots',
       'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq',
       'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star',
       'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space',
       'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token',
       'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port',
       'tld_in_path', 'tld_in_subdomain', 'nb_subdomains',
       'abnormal_subdomain', 'prefix_suffix', 'path_extension',
       'length_words_raw', 'char_repeat', 'shortest_word_host',
       'shortest_word_path', 'longest_words_raw', 'longest_word_host',
       'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path'
    ]]
X_html_test = df_con[[
        'phish_hints', 'domain_in_brand', 'nb_hyperlinks', 'ratio_intHyperlinks',
        'ratio_extHyperlinks', 'ratio_extRedirection', 'ratio_extErrors',
        'external_favicon', 'links_in_tags', 'ratio_extMedia', 'safe_anchor',
        'empty_title', 'domain_in_title', 'domain_with_copyright',
        'has_meta_refresh', 'has_js_redirect'
        ]]
X_ai_test = df_con[['creates_urgency', 'uses_threats', 'requests_sensitive_info',
            'offers_unrealistic_rewards', 'has_spelling_grammar_errors',
            'impersonated_brand', 'has_valid_copyright_year',
            'is_content_login_focused', 'has_rich_navigation',
            'has_physical_address', 'has_phone_number',
            'content_consistency_score', 'language_professionalism_score',
            'overall_phishing_likelihood_score', 'text_length'
            ]]
y = df_con[['target']]


In [29]:
X_ai_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32704 entries, 0 to 84119
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   creates_urgency                    32704 non-null  object 
 1   uses_threats                       32704 non-null  object 
 2   requests_sensitive_info            32704 non-null  object 
 3   offers_unrealistic_rewards         32704 non-null  object 
 4   has_spelling_grammar_errors        32704 non-null  object 
 5   impersonated_brand                 9009 non-null   object 
 6   has_valid_copyright_year           32704 non-null  object 
 7   is_content_login_focused           32704 non-null  object 
 8   has_rich_navigation                32704 non-null  object 
 9   has_physical_address               32704 non-null  object 
 10  has_phone_number                   32704 non-null  object 
 11  content_consistency_score          32704 non-null  float64


In [30]:
bool_col = ['creates_urgency', 'uses_threats', 'requests_sensitive_info',
            'offers_unrealistic_rewards', 'has_spelling_grammar_errors',
            'has_valid_copyright_year', 'is_content_login_focused',
            'has_rich_navigation', 'has_physical_address', 'has_phone_number']
X_ai_test[bool_col] = X_ai_test[bool_col].astype(int)
X_ai_test['impersonated_brand'] = [1 if k else 0 for k in X_ai_test['impersonated_brand'].notnull()]

/tmp/ipykernel_2385/1533107422.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_ai_test[bool_col] = X_ai_test[bool_col].astype(int)
/tmp/ipykernel_2385/1533107422.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_ai_test['impersonated_brand'] = [1 if k else 0 for k in X_ai_test['impersonated_brand'].notnull()]


In [31]:
X_ai_test

,creates_urgency,uses_threats,requests_sensitive_info,offers_unrealistic_rewards,has_spelling_grammar_errors,impersonated_brand,has_valid_copyright_year,is_content_login_focused,has_rich_navigation,has_physical_address,has_phone_number,content_consistency_score,language_professionalism_score,overall_phishing_likelihood_score,text_length
0,0,0,0,0,1,0,0,0,0,0,0,3.0,4.0,3.0,8934
1,0,0,0,0,0,1,0,0,1,1,0,8.0,10.0,2.0,3673
2,0,0,0,0,0,0,1,0,1,1,0,8.0,10.0,2.0,3095
5,0,0,0,0,0,0,1,0,1,0,0,9.0,10.0,2.0,3681
6,0,0,0,0,0,1,1,0,1,0,0,10.0,10.0,2.0,1515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62709,0,0,0,0,0,1,0,0,1,1,1,8.0,10.0,2.0,553
63871,0,0,1,0,0,0,0,1,1,0,0,6.0,8.0,6.0,2613
73630,0,1,0,0,0,0,0,0,0,0,0,3.0,8.0,5.0,297
83851,0,0,0,0,0,0,0,0,0,0,0,6.0,8.0,2.0,186


In [32]:
y_pred_proba_url = xgb_url_model.predict_proba(X_url_test)[:, 1]
y_pred_proba_html = mlp_html_model.predict(X_html_test).ravel()
y_pred_proba_ai = xgb_ai_model.predict_proba(X_ai_test)[:, 1]

print('url', len(y_pred_proba_url))
print('html', len(y_pred_proba_html))
print('ai',len(y_pred_proba_ai))

1022/1022 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
url 32704
html 32704
ai 32704


In [33]:
print('url', y_pred_proba_url)
print('html', y_pred_proba_html)
print('ai',y_pred_proba_ai)

url [0.99128306 0.983368   0.9906523  ... 0.11391444 0.02960393 0.08967892]
html [7.6895177e-01 5.3941834e-01 5.3449148e-01 ... 2.8929638e-03 1.7219529e-04
 6.0720090e-03]
ai [0.03421834 0.02307163 0.01924068 ... 0.32648394 0.45589566 0.06460512]


In [34]:
import numpy as np

In [35]:
X_meta = np.c_[y_pred_proba_url, y_pred_proba_html, y_pred_proba_ai]
y_meta = y

In [56]:
X_train_meta, X_test_meta, y_train_meta, y_test_meta = train_test_split(
    X_meta,
    y_meta,
    test_size=0.3,
    random_state=42,
    stratify=y_meta
)

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [58]:
len(y_train_meta[y_train_meta['target'] == 1])

20528

In [59]:
meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_train_meta, y_train_meta)

# 5. 評估元模型
final_score = meta_model.score(X_test_meta, y_test_meta)
print(f"最終堆疊模型的準確率: {final_score}")

最終堆疊模型的準確率: 0.9991846718304117


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [60]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [62]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, log_loss

# 1. 取得 "最終測試集" (X_test_meta, y_test_meta) 上的預測
y_pred = meta_model.predict(X_test_meta)
y_proba = meta_model.predict_proba(X_test_meta)[:, 1] # 取得機率

# 2. (必看) 混淆矩陣
print("--- 混淆矩陣 ---")
cm = confusion_matrix(y_test_meta, y_pred)
print(cm)
# [[TN, FP],
#  [FN, TP]]

# 3. (必看) 分類報告 (包含 Precision, Recall, F1)
print("\n--- 分類報告 ---")
report = classification_report(y_test_meta, y_pred, target_names=['安全 (0)', '釣魚 (1)'])
print(report)

# 4. (必看) ROC AUC
auc_score = roc_auc_score(y_test_meta, y_proba)
print(f"\nROC AUC 分數: {auc_score:.4f}")

# 5. (可選) Log Loss
log_loss_score = log_loss(y_test_meta, y_proba)
print(f"Log Loss 分數: {log_loss_score:.4f}")

--- 混淆矩陣 ---
[[1005    8]
 [   0 8799]]

--- 分類報告 ---
              precision    recall  f1-score   support

      安全 (0)       1.00      0.99      1.00      1013
      釣魚 (1)       1.00      1.00      1.00      8799

    accuracy                           1.00      9812
   macro avg       1.00      1.00      1.00      9812
weighted avg       1.00      1.00      1.00      9812


ROC AUC 分數: 0.9982
Log Loss 分數: 0.0128
